# VDJBet YF Analysis (Rmd-aligned)

Python rewrite of `tmp/vdjbet_snippet.Rmd` with matching analysis outputs:

1. YF vs OLGA V-usage and V/J correction factors
2. LLW reference and adjusted mock generation
3. LLW Pgen histogram match against mock bins
4. LLW overlap per sample: matched clonotypes and duplicate_count
5. Cohen d, z-scores, empirical p-values, FDR
6. Red line (LLW) + mock boxplots and Cohen d heatmaps

In [11]:
import math
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mir.basic.gene_usage import GeneUsage
from mir.basic.pgen import OlgaModel, PgenGeneUsageAdjustment
from mir.biomarkers.vdjbet import PgenBinPool, VDJBetOverlapAnalysis
from mir.common.parser import ClonotypeTableParser, load_vdjdb_latest
from mir.common.repertoire import LocusRepertoire, Repertoire

SEED = 42
N_MOCKS = 200
POOL_SIZE = 200_000
OLGA_USAGE_N = 100_000

_yfv_candidates = [
    Path("assets/large/yfv19"),
    Path("notebooks/assets/large/yfv19"),
    Path("../notebooks/assets/large/yfv19"),
]
YFV_DIR = next((p for p in _yfv_candidates if p.exists()), _yfv_candidates[0])

print(f"YFV_DIR = {YFV_DIR}")

ImportError: cannot import name 'PgenGeneUsageAdjustment' from 'mir.basic.pgen' (/Users/mikesh/vcs/mirpy/mir/basic/pgen.py)

## 1. Load LLWNGPMAV TRB reference from VDJdb

In [ ]:
vdjdb_rep = load_vdjdb_latest(
    epitope="LLWNGPMAV",
    locus="TRB",
    species="HomoSapiens",
    mhc_a_contains="A*02",
)
print(f"Reference clonotypes: {vdjdb_rep.clonotype_count}")
print(f"Example: {vdjdb_rep.clonotypes[0].junction_aa}  {vdjdb_rep.clonotypes[0].v_gene}  {vdjdb_rep.clonotypes[0].j_gene}")

## 2. Load YF samples

In [ ]:
if not (YFV_DIR / "metadata.txt").exists():
    raise FileNotFoundError(f"metadata.txt not found in {YFV_DIR}")

meta = pd.read_csv(YFV_DIR / "metadata.txt", sep="\t")
parser = ClonotypeTableParser()

samples = []
sample_dfs = []
for _, row in meta.iterrows():
    path = YFV_DIR / row["file_name"]
    if not path.exists():
        continue
    df = pd.read_csv(path, sep="\t", compression="infer")
    if "locus" in df.columns:
        df = df[df["locus"].fillna("") == "TRB"]
    df = df.dropna(subset=["junction_aa"])
    df = df[df["junction_aa"].astype(str).str.strip().str.len() > 0]

    clones = parser.parse_inner(df)
    rep = LocusRepertoire(clonotypes=clones, locus="TRB", repertoire_id=str(row["file_name"]))

    donor = str(row["donor"])
    day = int(row["day"])
    replica = str(row.get("replica", "F1"))

    samples.append({
        "file_name": str(row["file_name"]),
        "donor": donor,
        "day": day,
        "replica": replica,
        "repertoire": rep,
    })

    if {"v_gene", "j_gene", "duplicate_count"}.issubset(df.columns):
        sdf = df[["v_gene", "j_gene", "duplicate_count"]].copy()
    else:
        sdf = pd.DataFrame({
            "v_gene": [c.v_gene for c in clones],
            "j_gene": [c.j_gene for c in clones],
            "duplicate_count": [c.duplicate_count for c in clones],
        })

    sdf["v_base"] = sdf["v_gene"].fillna("").astype(str).str.split("*").str[0]
    sdf["j_base"] = sdf["j_gene"].fillna("").astype(str).str.split("*").str[0]
    sdf["duplicate_count"] = pd.to_numeric(sdf["duplicate_count"], errors="coerce").fillna(0).astype(float)
    sdf["donor"] = donor
    sdf["day"] = day
    sdf["replica"] = replica
    sample_dfs.append(sdf[["donor", "day", "replica", "v_base", "j_base", "duplicate_count"]])

if not samples:
    raise RuntimeError("No YF samples were loaded.")

yfv_table = pd.concat(sample_dfs, ignore_index=True)
combined = Repertoire(clonotypes=[c for s in samples for c in s["repertoire"].clonotypes], locus="TRB")
yfv_gu = GeneUsage.from_repertoire(combined)

print(f"Samples loaded: {len(samples)}")
print(f"Total TRB clonotypes across samples: {sum(len(s['repertoire'].clonotypes) for s in samples):,}")
print(f"Combined unique clonotypes object size: {combined.clonotype_count:,}")

## 3. OLGA usage and correction factors

Compute OLGA V and V/J usage, compare against YF usage, and derive correction factors.

- `factor_v = P_YF(V) / P_OLGA(V)`
- `factor_vj = P_YF(V,J) / P_OLGA(V,J)`

These factors are used by `PgenGeneUsageAdjustment`.

In [ ]:
olga_model = OlgaModel(locus="TRB", seed=SEED)
olga_gu = olga_model.compute_usage_cache(n=OLGA_USAGE_N, seed=SEED)

yfv_v = yfv_gu.v_usage("TRB")
olga_v = olga_gu.v_usage("TRB")

v_genes = sorted(set(yfv_v) | set(olga_v))
v_df = pd.DataFrame({
    "v_gene": v_genes,
    "p_yf": [float(yfv_v.get(g, 0.0)) for g in v_genes],
    "p_olga": [float(olga_v.get(g, 0.0)) for g in v_genes],
})
eps = 1e-9
v_df["factor_v"] = (v_df["p_yf"] + eps) / (v_df["p_olga"] + eps)
v_df["log2_factor_v"] = np.log2(v_df["factor_v"])

yfv_vj = yfv_gu.vj_usage("TRB")
olga_vj = olga_gu.vj_usage("TRB")
vj_keys = sorted(set(yfv_vj) | set(olga_vj))
vj_df = pd.DataFrame({
    "v_gene": [k[0] for k in vj_keys],
    "j_gene": [k[1] for k in vj_keys],
    "p_yf": [float(yfv_vj.get(k, 0.0)) for k in vj_keys],
    "p_olga": [float(olga_vj.get(k, 0.0)) for k in vj_keys],
})
vj_df["factor_vj"] = (vj_df["p_yf"] + eps) / (vj_df["p_olga"] + eps)
vj_df["log2_factor_vj"] = np.log2(vj_df["factor_vj"])

pgen_adj = PgenGeneUsageAdjustment(yfv_gu, cache_size=100_000, seed=SEED)

print("Top V genes by |log2 factor|:")
display(v_df.assign(abs_log2=lambda d: d["log2_factor_v"].abs()).sort_values("abs_log2", ascending=False).head(15))
print("Top VJ pairs by |log2 factor|:")
display(vj_df.assign(abs_log2=lambda d: d["log2_factor_vj"].abs()).sort_values("abs_log2", ascending=False).head(15))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.clip(v_df["p_yf"].values, 1e-8, None)
y = np.clip(v_df["p_olga"].values, 1e-8, None)
sc = axes[0].scatter(x, y, c=v_df["log2_factor_v"].values, cmap="RdBu_r", s=40, alpha=0.9)
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel("P_YF(V)")
axes[0].set_ylabel("P_OLGA(V)")
axes[0].set_title("V usage: YF vs OLGA")
axes[0].grid(alpha=0.2)
cb = plt.colorbar(sc, ax=axes[0], shrink=0.9)
cb.set_label("log2 correction factor")

top = v_df.assign(abs_log2=lambda d: d["log2_factor_v"].abs()).sort_values("abs_log2", ascending=False).head(20)
axes[1].barh(top["v_gene"], top["log2_factor_v"], color=["#c0392b" if z > 0 else "#2c3e50" for z in top["log2_factor_v"]])
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_title("Top V correction factors (log2)")
axes[1].set_xlabel("log2(P_YF(V) / P_OLGA(V))")

plt.tight_layout()
plt.show()

## 4. Build adjusted pool and compare LLW Pgen histogram to mock bins

In [ ]:
t0 = pd.Timestamp.now()
pool = PgenBinPool(
    "TRB",
    n=POOL_SIZE,
    n_jobs=4,
    seed=SEED,
    pgen_adjustment=pgen_adj,
)
analysis = VDJBetOverlapAnalysis(
    vdjdb_rep,
    pool=pool,
    n_mocks=N_MOCKS,
    seed=SEED,
)
dt = (pd.Timestamp.now() - t0).total_seconds()
print(f"Pool built: n_generated={pool.n_generated:,}, bins={len(pool.bins)}, floor={pool.floor_bin}, ceil={pool.ceil_bin}, elapsed={dt:.1f}s")

ref_bins = analysis.get_reference_bin_sample()
mock_bins = analysis.get_mock_bin_samples()

ref_counts = Counter(ref_bins)
all_bins = sorted(ref_counts.keys())
if not all_bins:
    raise RuntimeError("No LLW reference bins available for diagnostics.")

mock_mat = []
for mb in mock_bins:
    mc = Counter(mb)
    mock_mat.append([mc.get(b, 0) for b in all_bins])
mock_mat = np.asarray(mock_mat, dtype=float)

ref_vec = np.array([ref_counts.get(b, 0) for b in all_bins], dtype=float)
mock_mean = mock_mat.mean(axis=0)
mock_q10 = np.quantile(mock_mat, 0.10, axis=0)
mock_q90 = np.quantile(mock_mat, 0.90, axis=0)

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(all_bins, mock_q10, mock_q90, alpha=0.25, color="#1f77b4", label="Mock 10-90%")
ax.plot(all_bins, mock_mean, color="#1f77b4", linewidth=2, label="Mock mean")
ax.plot(all_bins, ref_vec, color="#d62728", marker="o", linewidth=2, label="LLW reference")
ax.set_xlabel("log2 Pgen bin")
ax.set_ylabel("count")
ax.set_title("LLW reference vs mock Pgen histogram")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

ref_prob = ref_vec / max(ref_vec.sum(), 1.0)
mock_probs = mock_mat / np.clip(mock_mat.sum(axis=1, keepdims=True), 1.0, None)
max_abs_diff = np.max(np.abs(mock_probs - ref_prob[None, :]), axis=1)
rmsd = np.sqrt(np.mean((mock_probs - ref_prob[None, :]) ** 2, axis=1))
print(f"Mock vs LLW: median max|Delta p(bin)|={np.median(max_abs_diff):.4f}, p95={np.percentile(max_abs_diff,95):.4f}")
print(f"Mock vs LLW: median RMSD={np.median(rmsd):.4f}, p95={np.percentile(rmsd,95):.4f}")

## 5. LLW overlap per sample: matched clonotypes and duplicate_count

Compute for each donor/day/replica:
- real LLW overlap
- mock distribution summary
- z-score and empirical p-value
- Cohen d
- FDR-adjusted p-values

In [ ]:
def bh_fdr(pvals):
    p = np.asarray(pvals, dtype=float)
    n = len(p)
    order = np.argsort(p)
    ranked = p[order]
    q = ranked * n / (np.arange(1, n + 1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0.0, 1.0)
    out = np.empty_like(q)
    out[order] = q
    return out

rows = []
for i, s in enumerate(samples, start=1):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        r = analysis.score(s["repertoire"], allow_1mm=False, match_v=True, match_j=True)

    mn = np.array(r.mock_n, dtype=float)
    mdc = np.array(r.mock_dc, dtype=float)
    mdc_log2 = np.log2(mdc + 1.0)
    real_dc_log2 = math.log2(r.dc + 1.0)

    mn_mean = float(np.mean(mn))
    mn_sd = float(np.std(mn, ddof=1)) if len(mn) > 1 else 0.0
    mdc_mean = float(np.mean(mdc_log2))
    mdc_sd = float(np.std(mdc_log2, ddof=1)) if len(mdc_log2) > 1 else 0.0

    z_n = (r.n - mn_mean) / mn_sd if mn_sd > 0 else 0.0
    z_dc = (real_dc_log2 - mdc_mean) / mdc_sd if mdc_sd > 0 else 0.0

    p_n_emp = (np.sum(mn >= r.n) + 1.0) / (len(mn) + 1.0)
    p_dc_emp = (np.sum(mdc_log2 >= real_dc_log2) + 1.0) / (len(mdc_log2) + 1.0)

    rows.append({
        "donor": s["donor"],
        "day": s["day"],
        "replica": s["replica"],
        "sample_label": f"{s['donor']} {s['replica']}",
        "n_total": r.n_total,
        "dc_total": r.dc_total,
        "matched_n_real": float(r.n),
        "matched_n_mock_mean": mn_mean,
        "matched_n_mock_sd": mn_sd,
        "matched_n_z": z_n,
        "matched_n_p_emp": p_n_emp,
        "matched_n_cohen_d": (r.n - mn_mean) / mn_sd if mn_sd > 0 else 0.0,
        "matched_dc_real": float(r.dc),
        "matched_dc_log2_real": real_dc_log2,
        "matched_dc_log2_mock_mean": mdc_mean,
        "matched_dc_log2_mock_sd": mdc_sd,
        "matched_dc_log2_z": z_dc,
        "matched_dc_log2_p_emp": p_dc_emp,
        "matched_dc_log2_cohen_d": (real_dc_log2 - mdc_mean) / mdc_sd if mdc_sd > 0 else 0.0,
        "mock_n": list(mn),
        "mock_dc_log2": list(mdc_log2),
    })

    if i % 10 == 0:
        print(f"Processed {i}/{len(samples)} samples")

df_res = pd.DataFrame(rows).sort_values(["donor", "replica", "day"]).reset_index(drop=True)
df_res["matched_n_p_adj"] = bh_fdr(df_res["matched_n_p_emp"].values)
df_res["matched_dc_log2_p_adj"] = bh_fdr(df_res["matched_dc_log2_p_emp"].values)

display(df_res[[
    "donor", "replica", "day",
    "matched_n_real", "matched_n_mock_mean", "matched_n_z", "matched_n_p_emp", "matched_n_cohen_d",
    "matched_dc_real", "matched_dc_log2_real", "matched_dc_log2_mock_mean", "matched_dc_log2_z", "matched_dc_log2_p_emp", "matched_dc_log2_cohen_d",
]])

## 6. Lines (LLW real) and boxplots (mock)

In [ ]:
plot_rows = []
for _, r in df_res.iterrows():
    for x in r["mock_n"]:
        plot_rows.append({
            "donor": r["donor"], "replica": r["replica"], "day": int(r["day"]),
            "metric": "matched_n", "kind": "mock", "value": float(x),
        })
    plot_rows.append({
        "donor": r["donor"], "replica": r["replica"], "day": int(r["day"]),
        "metric": "matched_n", "kind": "real", "value": float(r["matched_n_real"]),
    })

    for x in r["mock_dc_log2"]:
        plot_rows.append({
            "donor": r["donor"], "replica": r["replica"], "day": int(r["day"]),
            "metric": "matched_dc_log2", "kind": "mock", "value": float(x),
        })
    plot_rows.append({
        "donor": r["donor"], "replica": r["replica"], "day": int(r["day"]),
        "metric": "matched_dc_log2", "kind": "real", "value": float(r["matched_dc_log2_real"]),
    })

plot_df = pd.DataFrame(plot_rows)
days_all = sorted(df_res["day"].unique().tolist())
donors = sorted(df_res["donor"].unique().tolist())
replicas = sorted(df_res["replica"].unique().tolist())

def draw_panel(metric, ylabel, title):
    fig, axes = plt.subplots(len(replicas), len(donors), figsize=(4.0 * len(donors), 3.2 * len(replicas)), squeeze=False)
    fig.suptitle(title, fontsize=13, y=1.02)
    for ri, rep in enumerate(replicas):
        for di, donor in enumerate(donors):
            ax = axes[ri, di]
            sub = plot_df[(plot_df["metric"] == metric) & (plot_df["donor"] == donor) & (plot_df["replica"] == rep)]
            if sub.empty:
                ax.set_visible(False)
                continue

            real = sub[sub["kind"] == "real"].sort_values("day")
            mock = sub[sub["kind"] == "mock"]
            box_data = [mock[mock["day"] == d]["value"].values for d in days_all]
            width = 2.5
            ax.boxplot(
                box_data,
                positions=days_all,
                widths=width,
                patch_artist=True,
                boxprops=dict(facecolor="#d0e4f7", alpha=0.85),
                medianprops=dict(color="#1f77b4", linewidth=1.6),
                flierprops=dict(markersize=2),
                manage_ticks=False,
            )
            ax.plot(real["day"], real["value"], "-o", color="#d62728", linewidth=2, markersize=5, zorder=5)
            ax.set_xticks(days_all)
            ax.set_xlabel("Day")
            ax.set_ylabel(ylabel)
            ax.set_title(f"{donor} {rep}", fontsize=9)
            ax.grid(alpha=0.2)

    plt.tight_layout()
    plt.show()

draw_panel("matched_n", "Matched clonotypes", "LLW matched clonotypes: real (line) vs mock (boxplots)")
draw_panel("matched_dc_log2", "log2(matched duplicate_count + 1)", "LLW matched duplicate_count: real (line) vs mock (boxplots)")

## 7. Cohen d heatmaps

In [ ]:
def heatmap_cohen(value_col, p_adj_col, title, cbar_label, cmap="RdBu_r", vlim=None):
    work = df_res.copy()
    work["sample"] = work["donor"] + " " + work["replica"]
    pv = work.pivot_table(index="sample", columns="day", values=value_col, aggfunc="first")
    pp = work.pivot_table(index="sample", columns="day", values=p_adj_col, aggfunc="first")

    mat = pv.values.astype(float)
    if vlim is None:
        vmax = max(1.0, np.nanpercentile(np.abs(mat), 95))
        vmin = -vmax
    else:
        vmin, vmax = vlim

    fig, ax = plt.subplots(figsize=(8, max(3.5, 0.6 * pv.shape[0])))
    im = ax.imshow(mat, cmap=cmap, aspect="auto", vmin=vmin, vmax=vmax, interpolation="nearest")
    cb = plt.colorbar(im, ax=ax, shrink=0.85)
    cb.set_label(cbar_label)

    for r in range(pv.shape[0]):
        for c in range(pv.shape[1]):
            p = pp.values[r, c]
            d = pv.values[r, c]
            if pd.notna(p) and pd.notna(d) and (float(p) < 0.10) and (float(d) > 0):
                ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, fill=False, edgecolor="black", linewidth=2))

    ax.set_xticks(range(len(pv.columns)))
    ax.set_xticklabels([f"Day {d}" for d in pv.columns], rotation=45, ha="right")
    ax.set_yticks(range(len(pv.index)))
    ax.set_yticklabels(pv.index)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

heatmap_cohen(
    value_col="matched_n_cohen_d",
    p_adj_col="matched_n_p_adj",
    title="Matched clonotypes Cohen d (LLW vs mock)",
    cbar_label="Cohen d (matched clonotypes)",
    cmap="RdBu_r",
)

heatmap_cohen(
    value_col="matched_dc_log2_cohen_d",
    p_adj_col="matched_dc_log2_p_adj",
    title="Matched duplicate_count Cohen d (log2, LLW vs mock)",
    cbar_label="Cohen d (log2 matched duplicate_count)",
    cmap="Reds",
    vlim=(0, max(1.0, np.nanpercentile(df_res["matched_dc_log2_cohen_d"].values, 95))),
)

## 8. Final summary tables

In [ ]:
summary_cols = [
    "donor", "replica", "day",
    "matched_n_real", "matched_n_mock_mean", "matched_n_mock_sd", "matched_n_cohen_d", "matched_n_z", "matched_n_p_emp", "matched_n_p_adj",
    "matched_dc_real", "matched_dc_log2_real", "matched_dc_log2_mock_mean", "matched_dc_log2_mock_sd",
    "matched_dc_log2_cohen_d", "matched_dc_log2_z", "matched_dc_log2_p_emp", "matched_dc_log2_p_adj",
]
summary = df_res[summary_cols].copy()
for col in [
    "matched_n_real", "matched_n_mock_mean", "matched_n_mock_sd", "matched_n_cohen_d", "matched_n_z",
    "matched_dc_real", "matched_dc_log2_real", "matched_dc_log2_mock_mean", "matched_dc_log2_mock_sd",
    "matched_dc_log2_cohen_d", "matched_dc_log2_z",
]:
    summary[col] = summary[col].astype(float).round(3)
for col in ["matched_n_p_emp", "matched_n_p_adj", "matched_dc_log2_p_emp", "matched_dc_log2_p_adj"]:
    summary[col] = summary[col].map(lambda x: f"{x:.4f}")

display(summary.sort_values(["donor", "replica", "day"]).reset_index(drop=True))

print("Top positive matched clonotype effects by Cohen d:")
display(summary.sort_values("matched_n_cohen_d", ascending=False).head(12))

print("Top positive matched duplicate_count effects by Cohen d:")
display(summary.sort_values("matched_dc_log2_cohen_d", ascending=False).head(12))

Notebook output coverage checklist:

- V usage YF vs OLGA and correction factors
- LLW reference vs mock Pgen histogram alignment
- Matched clonotypes and duplicate_count per sample
- Cohen d, z-scores, empirical p-values, FDR
- Line + boxplot dynamics and Cohen d heatmaps

In [ ]:
print("Done: notebook rewritten to Rmd-aligned workflow.")
print(f"Samples: {len(samples)}, mocks: {N_MOCKS}, pool size: {POOL_SIZE:,}")
print("Use the summary table above for export/reporting.")